## Ch12. Deep Computer Vision Using Convolutional Neural Networks
- 지각은 쉬워 보이지만 의식 밖의 복잡한 처리라서 컴퓨터에게 어려운 문제였다.
- CNN은 시각 피질에서 영감을 얻었고, 컴퓨팅, 데이터, 11장의 훈련 기법이 맞물려 초인적 성능에 도달했다.

#### 시각 피질의 구조
- Hubel & Wiesel은 시각 피질 뉴런이 작은 국소 수용장에 반응하고, 위로 갈수록 저수준 패턴을 조합한 복잡한 패턴에 반응하는 계층 구조임을 밝힘
- 이것이 네오코그니트론(1980) -> LeNet-5(1998) -> 현대 CNN으로 이어졌고, 새 구성 요소는 합성곱층과 풀링층이다.
- 큰 이미지에서 완전 연결층은 파라미터가 너무 많아서, CNN은 부분 연결과 가중치 공유로 이를 해결한다.

#### 합성곱층
- 합성곱층 = 뉴런이 국소 수용장에만 연결된 부분 연결층이며, 층을 쌓으면 저수준 -> 고수준 특성으로 조립된다.
- CNN의 각 층은 2D라서 이미지를 평탄화하지 않는다.
- 제로 패딩은 출력 크기를 유지하고, 스트라이드는 수용장 간격을 넓혀 출력을 줄이고 계산량을 낮춘다.

#### 필터
- 필터(커널) = 뉴런의 가중치를 수용장 크기의 작은 이미지로 본 것이며, 특정 패턴(예: 세로선, 가로선)을 찾는 탐지기 역할을 한다.
- 같은 필터를 쓰는 뉴런들의 출력 = 특성 맵으로, 입력에서 해당 패턴이 있는 위치를 강조한다.
- 필터는 사람이 정하지 않고 훈련으로 자동 학습되며, 위층은 이를 조합해 복잡한 패턴을 만든다.

#### 여러 특성 맵 쌓기
- 합성곱층은 필터 여러개 -> 특성 맵 여러 개를 출력하는 3D 구조이다.
- 한 특성 맵 안의 뉴런은 같은 커널, 편향을 공유하므로 파라미터가 적고, 패턴을 위치와 무관하게 인식한다.
- 수용장은 이전 층의 모든 채널/특성 맵에 걸치며, 각 뉴런은 가중합 + 편향을 계산한다.
- 합성곱층 뉴런의 출력 = 수용장 안 모든 입력(모든 채널 포함)의 가중합 + 편향
- 가중치와 편향은 특성 맵 k에만 의존하므로 한 특성 맵의 모든 위치가 공유한다.
- 편향은 특성 맵 하나의 전체 밝기 조절 손잡이

#### PyTorch로 합성곱층 구현

In [3]:
# 샘플 이미지 준비
import numpy as np
import torch
from sklearn.datasets import load_sample_images
import torchvision
import torchvision.transforms.v2 as T

sample_images = np.stack(load_sample_images()["images"])
sample_images = torch.tensor(sample_images, dtype=torch.float32) / 255

print(sample_images.shape)
sample_images_permuted = sample_images.permute(0, 3, 1, 2)
print(sample_images_permuted.shape)
cropped_images = T.CenterCrop((70, 120))(sample_images_permuted)
print(cropped_images.shape)

torch.Size([2, 427, 640, 3])
torch.Size([2, 3, 427, 640])
torch.Size([2, 3, 70, 120])


In [5]:
# nn.Conv2d 만들기
import torch.nn as nn

torch.manual_seed(42)
conv_layer = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=7)
fmaps = conv_layer(cropped_images)
print(fmaps)
print(conv_layer.weight.shape)
print(conv_layer.bias.shape)

tensor([[[[-1.7997e-01, -4.7412e-02, -8.6408e-02,  ..., -5.4618e-01,
           -5.4526e-01, -5.4853e-01],
          [-1.3271e-01, -7.4603e-02, -7.7904e-02,  ..., -5.4617e-01,
           -5.4769e-01, -5.4878e-01],
          [-1.6372e-01, -6.5475e-02, -1.3059e-01,  ..., -5.4168e-01,
           -5.4184e-01, -5.4338e-01],
          ...,
          [-3.2100e-02, -6.4470e-02, -5.7826e-02,  ..., -5.1178e-01,
           -4.9536e-01, -4.8865e-01],
          [-2.0719e-02, -1.5698e-03,  2.2460e-02,  ..., -5.1116e-01,
           -5.0273e-01, -4.9861e-01],
          [ 1.4336e-01,  1.3034e-01,  1.0605e-01,  ..., -5.1006e-01,
           -4.9554e-01, -5.0024e-01]],

         [[-8.9471e-02, -1.9768e-01, -2.3245e-01,  ..., -6.2455e-01,
           -6.2589e-01, -6.2539e-01],
          [-1.9259e-01, -1.7489e-01, -3.3912e-01,  ..., -6.2293e-01,
           -6.2138e-01, -6.2281e-01],
          [-1.1531e-01, -1.6499e-01, -2.1657e-01,  ..., -6.2060e-01,
           -6.2136e-01, -6.1976e-01],
          ...,
     

#### 풀링층
- 풀링층은 가중치 없이 최댓값, 평균 같은 집계로 이미지를 축소해 계산량, 메모리, 파라미터를 줄인다.
- 최대 풀링(2x2, 스트라이드 2)은 높이, 너비를 절반으로 줄이고 채널 수는 유지하며, 작은 이동에 대한 불변성을 준다.
- 단점은 정보를 75%나 버리는 파괴성과, 분할처럼 위치가 중요한 작업에서는 불변성이 오히려 해롭다.

#### PyTorch로 풀링층 구현

In [6]:
max_pool = nn.MaxPool2d(kernel_size=2) # 최대 풀링

In [7]:
import torch.nn.functional as F

class DepthPool(torch.nn.Module):
	def __init__(self, kernel_size, stride=None, padding=0):
		super().__init__()
		self.kernel_size = kernel_size
		self.stride = stride if stride is not None else kernel_size
		self.padding = padding
	def forward(self, inputs):
		batch, channels, height, width = inputs.shape
		Z = inputs.view(batch, channels, height * width) # 공간 차원 합치기
		Z = Z.permute(0, 2, 1) # 공간과 채널 차원 교환
		Z = F.max_pool1d(Z, kernel_size=self.kernel_size, stride=self.stride, padding=self.padding) # 풀링
		Z = Z.permute(0, 2, 1) # 다시 교환
		return Z.view(batch, -1, height, width) # 공간 차원 복원

In [ ]:
# 전역 평균 풀링
global_avg_pool = nn.AdaptiveAvgPool2d(output_size=1)
output = global_avg_pool(cropped_images)
output = cropped_images.mean(dim=(2,3), keepdim=True)

#### CNN 아키텍처
- CNN = [합성곱 + ReLU]몇 개 -> 풀링을 반복(공간은 줄고 깊이는 늘어남) + 위에 완전연결 분류기.
- 일반 층은 3x3커널, 첫 층만 큰 커널(+큰 스트라이드)도 가능, 풀링 뒤 필터 수 2배가 관행
- Flatten 직전의 특성 수는 채널x높이x너비

In [ ]:
from functools import partial

DefaultConv2d = partial(nn.Conv2d, kernel_size=3, padding="same")
model = nn.Sequential(DefaultConv2d(in_channels=1, out_channels=64, kernel_size=7), nn.ReLU(),
                      nn.MaxPool2d(kernel_size=2), DefaultConv2d(in_channels=64, out_channels=128), nn.ReLU(),
					  DefaultConv2d(in_channels=128, out_channels=128), nn.ReLU(),
					  nn.MaxPool2d(kernel_size=2),
					  DefaultConv2d(in_channels=128, out_channels=256), nn.ReLU(), nn.MaxPool2d(kernel_size=2),
					  DefaultConv2d(in_channels=128, out_channels=256), nn.ReLU(),
					  DefaultConv2d(in_channels=256, out_channels=256), nn.ReLU(),
					  nn.MaxPool2d(kernel_size=2), nn.Flatten(), nn.Linear(in_features=2304, out_features=128),
					  nn.ReLU(), nn.Dropout(0.5), nn.Linear(in_features=128, out_features=64), nn.ReLU(),
					  nn.Dropout(0.5), nn.Linear(in_features=64, out_features=10),).to(device)

#### LeNet-5
- LeNet-5 = conv(6) -> pool -> conv(16) -> pool -> conv(120) -> FC(84) -> 출력(10)
- 이미 '합성곱+풀링 반복, 뒤에 밀집층' 이라는 현대 CNN의 기본 틀을 갖추고 있었다.
- 현대와의 핵심 차이는 활성화(tanh->ReLU)와 출력(RBF->softmax)

#### AlexNet
- ILSVRC 2012에서 오류율 17% 대 26%로 압승하며 딥러닝 시대를 연 모델
- LeNet-5와 같은 틀이지만 ReLU, 최대 풀링, 드롭아웃, 데이터 증강을 쓰고 규모가 훨씬 크다.
- 데이터 증강: 현실적인 변형본으로 데이터를 늘려 과대적합을 줄이는 규제 기법

#### GoogLeNet
- 인셉션 모듈 9개를 쌓은 깊은 네트워크이며, 파라미터는 AlexNet의 약 1/10(약 600만)

#### ResNet
- 핵심은 스킵 연결로, 네트워크가 h(x) 대신 잔차 f(x) = h(x) - x를 학습한다.
- 초기에 항등 함수처럼 동작해서 훈련이 빠르고, 일부 층이 학습 전이라도 신호가 전달된다.

#### Xception
- 인셉션 모듈을 분리 합성곱으로 대체한 모델
- 분리 합성곱 = 깊이별(채널마다 공간 필터) + 점별(1x1, 채널 간 혼합)

In [ ]:
# Xception

class SeparableConv2d(nn.Module):
	def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
		super().__init__()
		self.depthwise_conv = nn.Conv2d(
			in_channels, in_channels, kernel_size, stride=stride, padding=padding,
			groups=in_channels
		)
		self.pointwise_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0)
		def forward(self, inputs):
			return self.pointwise_conv(self.depthwise_conv(inputs))

#### SENet
- 기존 모듈마다 SE블록을 붙인 구조
- SE 블록 = 전역 평균 풀링 -> 작은 밀집층(ReLU) -> 밀집층(sigmoid)으로 채널별 가중치(0~1)를 만들어 곱함

#### 기타 주목할 만한 CNN 아키텍처
- VGG(3x3 대량 적층), ResNeXt(병렬 스택), DenseNet(전 층 연결), MobileNet(경량), CSPNet, ConvNeXt(트랜스포머식 ResNet)가 주요 변형

#### CNN 아키텍처 고르기
- 아키텍처 선택은 정확도 / 크기 / 속도 / 에너지 중 우선순위에 달려 있다.
- EfficientNet v2 계열과 ConvNeXt가 정확도가 높고, MobileNet v3 small과 EfficientNet B0는 가볍다.

#### GPU RAM 요구량: 추론 vs 훈련
- 합성곱층은 파라미터는 적지만 활성화 메모리와 연산량이 크다.
- 추론은 인접한 두 층만, 훈련은 모든 층의 활성화를 저장해야 해서 OOM이 잘 난다.
- 대처: 배치 축소, 그래디언트 누적, 16비트 float, 분산, CPU 오프로드, 체크포인팅
- 더 효율적인 대안: 가역 잔차 네트워크

#### 가역 잔차 네트워크 (RevNet)
- 가역 층: y_1 = x_1 + f(x_2), y_2 = g(y_1) + x_2
- 출력에서 입력을 복원하므로 순전파 활성화를 저장할 필요가 없고, 연산은 약 33% 증가
- 스트라이드 > 1이나 valid 패딩 층은 가역이 안 되어 활성화를 저장해야 하며, 보통 소수이다.
- Reformer 같은 트랜스포머에도 쓰인다.

In [ ]:
# ResNet-34 PyTorch 구현
# ResidualUnit
class ResidualUnit(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        DefaultConv2d = partial(
            nn.Conv2d, kernel_size=3, stride=1, padding=1, bias=False)
        self.main_layers = nn.Sequential(
            DefaultConv2d(in_channels, out_channels, stride=stride),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            DefaultConv2d(out_channels, out_channels),
            nn.BatchNorm2d(out_channels),
        )
        if stride > 1:
            self.skip_connection = nn.Sequential(
                DefaultConv2d(in_channels, out_channels, kernel_size=1,
                              stride=stride, padding=0),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.skip_connection = nn.Identity()

    def forward(self, inputs):
        return F.relu(self.main_layers(inputs) + self.skip_connection(inputs))

	# ResNet34
class ResNet34(nn.Module):
    def __init__(self):
        super().__init__()
        layers = [
            nn.Conv2d(in_channels=3, out_channels=64, kernel_size=7, stride=2,
                      padding=3, bias=False),
            nn.BatchNorm2d(num_features=64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        ]
        prev_filters = 64
        for filters in [64] * 3 + [128] * 4 + [256] * 6 + [512] * 3:
            stride = 1 if filters == prev_filters else 2
            layers.append(ResidualUnit(prev_filters, filters, stride=stride))
            prev_filters = filters
        layers += [
            nn.AdaptiveAvgPool2d(output_size=1),
            nn.Flatten(),
            nn.LazyLinear(10),
        ]
        self.resnet = nn.Sequential(*layers)

    def forward(self, inputs):
        return self.resnet(inputs)

In [ ]:
# TorchVision 사전훈련 모델 사용
# 모델 불러오기
weights = torchvision.models.ConvNeXt_Base_Weights.IMAGENET1K_V1
model = torchvision.models.convnext_base(weights=weights).to(device)
# 전처리
transforms = weights.transforms()
preprocessed_images = transforms(sample_images_permuted)
# 예측
model.eval()
with torch.no_grad():
	y_logits = model(preprocessed_images.to(device))

y_pred = torch.argmax(y_logtis, dim=1)
print(y_pred)

# 클래스 이름 얻기
class_names = weights.meta["categories"]
[class_names[class_id] for class_id in y_pred]

# 상위 3개 예측
y_top3_logits, y_top3_class_ids = y_logits.topk(k=3, dim=1)
[[class_names[class_id] for class_id in top3] for top3 in y_top3_class_ids]
print(y_top3_logits.softmax(dim=1))

In [ ]:
# 전이 학습용 사전훈련 모델
# 데이터 준비
from torch.utils.data import DataLoader
DefaultFlowers102 = partial(torchvision.datasets.Flowers102, root="datasets",
                            transform=weights.transforms(), download=True)
train_set = DefaultFlowers102(split="train")
valid_set = DefaultFlowers102(split="val")
test_set = DefaultFlowers102(split="test")

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=32)
test_loader = DataLoader(test_set, batch_size=32)

class_names =  ['pink primrose', ..., 'trumpet creeper', 'blackberry lily']
print([name for name, child in model.named_children()])
print(model.classifier)

n_classes = 102
model.classifier[2] = nn.Linear(1024, n_classes).to(device)
# 동결
for param in model.parameters():
	param.requires_grad = False
for param in model.classifier.parameters():
	param.requires_grad = True

In [ ]:
# 데이터 증강으로 정확도 높이기
import torchvision.transforms.v2 as T

transforms = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=30),
    T.RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0)),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

#### 분류와 위치 탐색
- 위치 탐색 = 박스 4개 숫자를 회귀. 모델에 두 번째 헤드(Linear->4)만 추가하고, 손실은 CrossEntropy + 위치 손실의 가중합

In [ ]:
# FlowerLocator
class FlowerLocator(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
        self.localization_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(base_model.classifier[2].in_features, 4)
        )

    def forward(self, X):
        features = self.base_model.features(X)
        pool = self.base_model.avgpool(features)
        logits = self.base_model.classifier(pool)
        bbox = self.localization_head(pool)
        return logits, bbox

torch.manual_seed(42)
locator_model = FlowerLocator(model).to(device)

In [ ]:
# 바운딩 박스 다루기
import torchvision.tv_tensors

bbox = torchvision.tv_tensors.BoundingBoxes(
    [[377, 199, 248, 262]],  # center x=377, center y=199, width=248, height=262
    format="CXCYWH",         # 다른 형식: "XYXY", "XYWH"
    canvas_size=(500, 754)   # 전처리 전 원본 이미지 크기
)

#### 객체 탐지
- 여러 객체의 분류 + 위치 탐색. 슬라이딩 CNN이 위치마다 클래스 확률, 박스, 객체성 점수를 예측
- 크기가 다양한 객체를 위해 여러 영영 크기로 훑음
- 중목 박스는 NMS(임계값 미만 제거 -> 최고점 박스와 IoU 큰 박스 제거 -> 반복)로 정리
- 문제는 속도이고, FCN이 해결책이다.

#### 완전 합성곱 신경망 (FCN)
- 밀집층은 필터 수 = 뉴런 수, 필터 크기 = 입력 특성 맵 크기, valid 패딩의 합성곱층으로 바꿀 수 있고 결과가 같다.
- 합성곱은 입력 크기에 제한이 없어 FCN은 어떤 크기의 이미지도 처리한다.
- 큰 이미지를 넣으면 출력이 격자가 되어, 슬라이딩 CNN을 한 번의 순전파로 대체한다.

#### YOLO
- FCN식으로 한 번의 순전파에 격자 예측을 내며, 셀 상대 좌표, 셀당 박스 2개(각각 객체성), 셀당 클래스 분포가 특징
- mAP = 재현율 구간별 최대 정밀도의 평균(AP)을 클래스별로 평균한 것. 탐지에서는 IoU 임계값을 함께 쓰며 COCO는 0.50~0.95를 모두 평균한다.

#### 객체 추적
- 추적 = 프레임마다 탐지한 객체를 시간에 따라 같은 객체로 연결하는 일

In [ ]:
# Ultralytics의 추적
my_video = "https://homl.info/cars.mp4"
results = model.track(source=my_video, stream=True, save=True)
for frame_results in results:
    summary = frame_results.summary()  # 앞선 summary + track id
    track_ids = [obj["track_id"] for obj in summary]
    print("Track ids:", track_ids)

#### 시맨틱 분할
- 시맨틱 분할 = 픽셀 단위 분류. 해상도 손실이 핵심 문제
- 해법: CNN을 FCN으로 바꾸고 전치 합성곱으로 업샘플링 + 아래층 스킵 연결로 세부 해상도 복구
- 인스턴스 분할은 같은 클래스의 객체를 개별로 구분한다.

#### Exercises
1. CNN이 이미지 분류에서 완전 연결 DNN보다 나은 점
- CNN은 가중치 공유, 지역 연결로 파라미터를 크게 줄이고, 물체의 위치가 달라져도 인식하며, 이미지의 공간 구조와 계층적 특징을 활용하기 때문에 더 효율적이고 정확함

2. 
- 파라미터 = (커널 높이 x 너비 x 입력 채널 + 1) x 출력 채널
- 출력 크기 = ceil(입력 / 2)

3. GPU 메모리 부족(OOM) 시 시도할 수 있는 5가지
- 배치 크기 줄이기, 16비트 float 사용, 활성화 체크포인팅, 모델 줄이기(차원 축소), 여러 장치로 분산하거나 CPU로 오프로드

4. 스트라이드가 같은 합성곱층 대신 최대 풀링층을 쓰는 이유
- 최대 풀링의 장점은 파라미터가 없고, 계산이 가볍고, 작은 이동에 대한 불변성이 있고, 가장 강한 활성화를 보존한다.

5. 아키텍처별 주요 혁신
- AlexNet: 깊고 큰 CNN + ReLU, 드롭아웃, 데이터 증강, LRN. 합성곱층을 풀링 없이 연달아 쌓은 최초의 모델
- GoogLeNet: 인셉션 모듈(여러 커널 병렬) + 1×1 병목 + 전역 평균 풀링으로 파라미터를 AlexNet의 1/10로 감소
- ResNet: 스킵 연결(잔차 학습)로 152층 같은 초심층 네트워크 훈련 가능
- SENet: SE 블록으로 채널별 중요도를 재보정하는 채널 어텐션. 기존 모듈에 추가하는 방식
- Xception: 깊이별 분리 합성곱(공간 패턴과 채널 패턴을 분리)으로 인셉션 모듈 대체
- EfficientNet: 복합 스케일링으로 깊이, 너비, 해상도를 함께 키움 (α·β²·γ² ≈ 2)
- ConvNeXt: 비전 트랜스포머 기법(큰 커널, 활성화/정규화 층 감소)을 적용한 현대화된 ResNet

6. 완전 합성곱 신경망 (FCN)이란
- 합성곱층과 풀링층만으로 이루어진 신경망. 밀집층이 하나도 없다.

7. 시맨틱 분할의 주된 기술적 어려움
- 깊은 CNN은 풀링과 스트라이드로 위치 정보(해상도)를 잃는데, 분할은 픽셀 단위의 정밀한 위치가 필요하다는 것이 주된 어려움

In [ ]:
# Exercises 8. CNN을 처음부터 직접 구축하고 MNIST 데이터셋에서 최대한 높은 정확도 달성

import argparse, copy, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms.v2 as T
from torch.utils.data import DataLoader, Subset
 
parser = argparse.ArgumentParser()
parser.add_argument("--epochs", type=int, default=20)
parser.add_argument("--batch_size", type=int, default=128)
parser.add_argument("--lr", type=float, default=3e-3)  # OneCycle의 최대 학습률
parser.add_argument("--workers", type=int, default=2)
args = parser.parse_args()
 
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
np.random.seed(42)
print("device:", device)
 
# 1. 데이터
MEAN, STD = 0.1307, 0.3081  # MNIST 훈련 세트의 평균/표준편차
 
# 증강은 훈련 세트에만, 정규화는 모두에게 (정규화는 항상 마지막)
train_tf = T.Compose([
    T.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    T.ToImage(), T.ToDtype(torch.float32, scale=True), T.Normalize([MEAN], [STD]),
])
eval_tf = T.Compose([
    T.ToImage(), T.ToDtype(torch.float32, scale=True), T.Normalize([MEAN], [STD]),
])
 
# 같은 60,000장을 두 가지 변환으로 열고, 인덱스로 훈련 55,000 / 검증 5,000 분할
full_train = torchvision.datasets.MNIST("datasets", train=True, download=True, transform=train_tf)
full_valid = torchvision.datasets.MNIST("datasets", train=True, download=True, transform=eval_tf)
test_set = torchvision.datasets.MNIST("datasets", train=False, download=True, transform=eval_tf)
 
perm = torch.randperm(len(full_train)).tolist()
train_set = Subset(full_train, perm[5000:])
valid_set = Subset(full_valid, perm[:5000])
 
pin = device == "cuda"
train_loader = DataLoader(train_set, args.batch_size, shuffle=True,
                          num_workers=args.workers, pin_memory=pin, drop_last=True)
valid_loader = DataLoader(valid_set, 1000, num_workers=args.workers, pin_memory=pin)
test_loader = DataLoader(test_set, 1000, num_workers=args.workers, pin_memory=pin)
print("train/valid/test:", len(train_set), len(valid_set), len(test_set))
 
# 2. 모델
def conv_bn(c_in, c_out):
    # BN이 뒤에 오므로 bias=False
    return [nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
            nn.BatchNorm2d(c_out), nn.ReLU()]
 
def block(c_in, c_out, drop):
    return nn.Sequential(*conv_bn(c_in, c_out), *conv_bn(c_out, c_out),
                         nn.MaxPool2d(2), nn.Dropout(drop))
 
model = nn.Sequential(
    block(1, 32, 0.1),     # 28 -> 14
    block(32, 64, 0.2),    # 14 -> 7
    block(64, 128, 0.3),   # 7 -> 3
    nn.AdaptiveAvgPool2d(1), nn.Flatten(),
    nn.Linear(128, 10),    # 로짓 (softmax는 손실 함수 안에 포함)
).to(device)
print("params:", sum(p.numel() for p in model.parameters()))
 
# 3. 훈련
@torch.no_grad()
def evaluate(loader):
    model.eval()
    loss = correct = n = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        out = model(X)
        loss += F.cross_entropy(out, y, reduction="sum").item()
        correct += (out.argmax(1) == y).sum().item()
        n += len(y)
    return loss / n, correct / n
 
opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=args.lr, total_steps=args.epochs * len(train_loader))
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
 
best_acc, best_state = 0.0, None
for epoch in range(args.epochs):
    t0 = time.time()
    model.train()
    tot = 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        loss = criterion(model(X), y)
        loss.backward()
        opt.step(); opt.zero_grad()
        sched.step()
        tot += loss.item() * len(y)
    vl, va = evaluate(valid_loader)
    print(f"epoch {epoch+1:2d}  train {tot / (len(train_loader) * args.batch_size):.4f}  "
          f"valid {vl:.4f}  acc {va:.4f}  ({time.time() - t0:.1f}s)")
    if va > best_acc:
        best_acc, best_state = va, copy.deepcopy(model.state_dict())
 
# 4. 최종 평가
model.load_state_dict(best_state)
_, test_acc = evaluate(test_loader)
print(f"best valid acc = {best_acc:.4f}   test acc = {test_acc:.4f}")
torch.save(best_state, "mnist_cnn.pt")

In [ ]:
# Exercises 9. transfer_learning

import argparse, copy, random, time, zipfile, urllib.request
from pathlib import Path
import numpy as np
import torch
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms.v2 as T
from torch.utils.data import DataLoader, Subset
 
parser = argparse.ArgumentParser()
parser.add_argument("--data_dir", type=str, default=None)
parser.add_argument("--download", action="store_true", help="PyTorch 튜토리얼 개미/벌 데이터 사용")
parser.add_argument("--model", default="efficientnet_b0",
                    choices=["efficientnet_b0", "mobilenet_v3_small", "resnet34", "convnext_tiny"])
parser.add_argument("--no_pretrained", action="store_true", help="가중치 다운로드 생략(코드 점검용)")
parser.add_argument("--head_epochs", type=int, default=5)
parser.add_argument("--ft_epochs", type=int, default=10)
parser.add_argument("--batch_size", type=int, default=32)
parser.add_argument("--lr", type=float, default=1e-3)       # 헤드 훈련용. 파인튜닝은 1/10
parser.add_argument("--workers", type=int, default=2)
parser.add_argument("--seed", type=int, default=42)
args = parser.parse_args()
 
if torch.cuda.is_available(): device = "cuda"
elif hasattr(torch, "xpu") and torch.xpu.is_available(): device = "xpu"   # Intel GPU
else: device = "cpu"
random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
print("device:", device)
 
# 1. 데이터
if args.download:
    root = Path("datasets"); root.mkdir(exist_ok=True)
    zpath = root / "hymenoptera_data.zip"
    if not zpath.exists():
        urllib.request.urlretrieve("https://download.pytorch.org/tutorial/hymenoptera_data.zip", zpath)
    with zipfile.ZipFile(zpath) as z: z.extractall(root)
    # 튜토리얼 데이터는 train/val로 나뉘어 있으므로 합쳐서 직접 3분할한다
    data_dir = root / "hymenoptera_data"
    merged = {}
    for split in ("train", "val"):
        for cdir in (data_dir / split).iterdir():
            merged.setdefault(cdir.name, []).extend(sorted(cdir.glob("*.jpg")))
    paths_by_class = merged
    class_names = sorted(merged)
    samples = [(str(p), i) for i, c in enumerate(class_names) for p in merged[c]]
elif args.data_dir:
    ds0 = torchvision.datasets.ImageFolder(args.data_dir)
    class_names, samples = ds0.classes, ds0.samples
else:
    raise SystemExit("--data_dir 폴더를 지정하거나 --download 를 사용하세요.")
 
n_classes = len(class_names)
counts = np.bincount([y for _, y in samples], minlength=n_classes)
print("classes:", dict(zip(class_names, counts.tolist())))
if counts.min() < 100:
    print("경고: 클래스당 100장 미만인 클래스가 있습니다.")
 
# 2. 층화 분할 (클래스별 70/15/15)
rng = random.Random(args.seed)
tr_idx, va_idx, te_idx = [], [], []
for c in range(n_classes):
    idx = [i for i, (_, y) in enumerate(samples) if y == c]
    rng.shuffle(idx)
    n_va = n_te = max(1, int(0.15 * len(idx)))
    te_idx += idx[:n_te]; va_idx += idx[n_te:n_te + n_va]; tr_idx += idx[n_te + n_va:]
print(f"train/valid/test: {len(tr_idx)}/{len(va_idx)}/{len(te_idx)}")
 
# 3. 입력 파이프라인
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]  # ImageNet 통계 (사전훈련 때와 동일해야 함)
norm = [T.ToImage(), T.ToDtype(torch.float32, scale=True), T.Normalize(MEAN, STD)]
train_tf = T.Compose([  # 증강은 훈련 세트에만, 정규화 이전에
    T.RandomResizedCrop(224, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(0.2, 0.2, 0.2, 0.05),
    *norm])
eval_tf = T.Compose([T.Resize(256), T.CenterCrop(224), *norm])
 
class ImgDataset(torch.utils.data.Dataset):
    def __init__(self, items, tf): self.items, self.tf = items, tf
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        path, y = self.items[i]
        return self.tf(Image.open(path).convert("RGB")), y
 
pin = device == "cuda"
mk = lambda idx, tf, shuffle: DataLoader(
    ImgDataset([samples[i] for i in idx], tf), args.batch_size, shuffle=shuffle,
    num_workers=args.workers, pin_memory=pin)
train_loader, valid_loader, test_loader = mk(tr_idx, train_tf, True), mk(va_idx, eval_tf, False), mk(te_idx, eval_tf, False)
 
# 4. 모델: 헤드 교체
def build_model():
    ctor = getattr(torchvision.models, args.model)
    weights = None if args.no_pretrained else "DEFAULT"
    model = ctor(weights=weights)
    # 마지막 nn.Linear(출력층)를 찾아 클래스 수에 맞게 교체
    name, last = [(n, m) for n, m in model.named_modules() if isinstance(m, nn.Linear)][-1]
    parent = model
    *path, leaf = name.split(".")
    for p in path: parent = getattr(parent, p) if not p.isdigit() else parent[int(p)]
    new = nn.Linear(last.in_features, n_classes)
    if leaf.isdigit(): parent[int(leaf)] = new
    else: setattr(parent, leaf, new)
    head_prefix = name.rsplit(".", 1)[0] if "." in name else name  # 예: "classifier" 또는 "fc"
    return model.to(device), head_prefix
 
model, head_prefix = build_model()
head_params = [p for n, p in model.named_parameters() if n.startswith(head_prefix)]
print(f"model={args.model}  head='{head_prefix}'  params={sum(p.numel() for p in model.parameters())/1e6:.1f}M")
 
# 5. 훈련 / 평가
@torch.no_grad()
def evaluate(loader):
    model.eval()
    loss = correct = n = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        out = model(X)
        loss += F.cross_entropy(out, y, reduction="sum").item()
        correct += (out.argmax(1) == y).sum().item()
        n += len(y)
    return loss / n, correct / n
 
def fit(params, lr, epochs, tag):
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, epochs))
    best_acc, best_state = -1, None
    for epoch in range(epochs):
        t0 = time.time(); model.train(); tot = 0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            loss = F.cross_entropy(model(X), y)
            loss.backward(); opt.step(); opt.zero_grad()
            tot += loss.item() * len(y)
        sched.step()
        vl, va = evaluate(valid_loader)
        print(f"[{tag}] epoch {epoch+1:2d}  train {tot/len(tr_idx):.4f}  valid {vl:.4f}  acc {va:.4f}  ({time.time()-t0:.0f}s)")
        if va > best_acc: best_acc, best_state = va, copy.deepcopy(model.state_dict())
    if best_state: model.load_state_dict(best_state)
    return best_acc
 
# 단계 1: 사전훈련 층 동결, 헤드만 훈련
for p in model.parameters(): p.requires_grad = False
for p in head_params: p.requires_grad = True
a1 = fit(head_params, args.lr, args.head_epochs, "head")
 
# 단계 2: 전체 해동, 학습률 1/10로 파인튜닝
for p in model.parameters(): p.requires_grad = True
a2 = fit(model.parameters(), args.lr / 10, args.ft_epochs, "finetune")
 
_, test_acc = evaluate(test_loader)
print(f"valid acc: head-only {a1:.4f} -> fine-tuned {a2:.4f}   |   TEST acc = {test_acc:.4f}")
torch.save({"state": model.state_dict(), "classes": class_names, "model": args.model}, "transfer_model.pt")